# Multisine Signal Tone and Power Harvesting Measurements across Gain and Distance Settings Exploration with `mlcroissant`
This notebook demonstrates how to load, examine, and analyze the FAIR² wireless power transfer dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described and linked via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.gk8g-g8p8/fair2.json](https://sen.science/doi/10.71728/senscience.gk8g-g8p8/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.gk8g-g8p8/fair2.json'

# Load metadata via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print high-level description
print(f"Dataset Name: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)[:600]} ...\n")

## 2. Data Overview
The dataset has two record sets: the measurements (one row per sample: the receiver-side power at each of the twelve transmitter gain and distance settings, and the class label) and the partitioning of the twelve settings between model development and external testing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

record_sets = {rs.name: rs for rs in metadata.record_sets}
display(pd.DataFrame([{'Record set': name, 'Fields': len(rs.fields), 'Description': rs.description}
                      for name, rs in record_sets.items()])
        .style.hide(axis='index').set_caption('Record sets'))

## 3. Data Extraction
Load both record sets as pandas DataFrames, with the field names as columns (`mlcroissant` names them by field `@id`) and the text decoded. The measurements are then reshaped to one row per sample and setting, with the setting's gain, distance and partition. The class label `Indicator` stands for the number of tones of the multisine waveform: 0 = 2 tones, 1 = 4, 2 = 8, 3 = 16 and 4 = 32.

In [ ]:
def decode(value):
    return value.decode('utf-8') if isinstance(value, bytes) else value

def load(name):
    record_set = record_sets[name]
    df = pd.DataFrame(dataset.records(record_set=record_set.id))
    return df.rename(columns={field.id: field.name for field in record_set.fields}).apply(lambda column: column.map(decode))

measurements = load('multisine_signals_v1.0.0.csv')
settings = load('dataset_partitioning_v1.1.0.csv')

TONES = {0: 2, 1: 4, 2: 8, 3: 16, 4: 32}
GROUPS = {'A': 'A: model development', 'B': 'B: external test'}
measurements['tones'] = measurements['Indicator'].map(TONES)

power = (measurements.melt(id_vars=['ID', 'tones'], value_vars=list(settings['variable_name']),
                           var_name='setting', value_name='power_mW')
         .merge(settings[['variable_name', 'dataset_group', 'tx_gain_percent', 'tx_gain_db', 'distance_cm']],
                left_on='setting', right_on='variable_name').drop(columns='variable_name'))

display(pd.DataFrame({'Value': {
    'Samples': len(measurements),
    'Samples per class': measurements['tones'].value_counts().iloc[0],
    'Gain and distance settings': len(settings),
    'Power readings': len(power),
}}).style.format('{:,}').set_caption('Loaded data'))

## 4. Exploratory Data Analysis (EDA)
### 4.1 Measurement design
Each sample was measured at twelve settings of transmitter gain and distance, split in two groups: group **A** (high gain, short distances) for model development, and group **B** (lower or intermediate gain, longer distances) reserved for external testing. Gain and distance were not crossed: each gain was measured at three distances of one group only.

In [ ]:
design = settings.pivot_table(index=['tx_gain_percent', 'tx_gain_db'], columns='distance_cm',
                              values='dataset_group', aggfunc='first').sort_index(ascending=False)
mean_power = power.groupby(['tx_gain_percent', 'distance_cm'])['power_mW'].mean()
cells_text = design.copy().astype(object)
for (gain, gain_db), row in design.iterrows():
    for distance, group in row.items():
        cells_text.loc[(gain, gain_db), distance] = '' if pd.isna(group) else f'{group} · {mean_power[(gain, distance)]:.2f} mW'
cells_text.index.names = ['Gain (%)', 'Gain (dB)']
cells_text.columns = [f'{distance} cm' for distance in cells_text.columns]

def shade(value):
    return '' if value == '' else ('background-color: #cde2fb' if value.startswith('A') else 'background-color: #fde3d5')

display(cells_text.style.map(shade).format_index('{:.2f}', axis=0, level=1).set_caption('Settings measured: partition and mean power (all classes)'))

### 4.2 Grouping: power by number of tones
The median received power of each class at each setting. At every setting, the power falls as the number of tones increases: that is the signal a classifier can use.

In [ ]:
SETTING_ORDER = list(settings.sort_values(['dataset_group', 'tx_gain_percent', 'distance_cm'],
                                          ascending=[True, False, True])['variable_name'])
medians = power.pivot_table(index='tones', columns='setting', values='power_mW', aggfunc='median')[SETTING_ORDER]
medians.index.name = 'Tones'
medians.columns.name = None
display(medians.style.format('{:.2f}').background_gradient(cmap='Blues', axis=0)
        .set_caption('Median received power (mW) by number of tones and setting (group A first, then B)'))

### 4.3 Normalizing: power relative to two tones
The settings differ by more than an order of magnitude in power. Dividing each class's median power by the median power with two tones at the same setting removes that difference and keeps the shape of the drop. Averaged within each group:

In [ ]:
relative = medians.div(medians.loc[2], axis=1)
group_of = settings.set_index('variable_name')['dataset_group']
relative_by_group = relative.T.groupby(group_of).mean().T.rename(columns=GROUPS)
relative_by_group.columns.name = None
display(relative_by_group.style.format('{:.2f}').background_gradient(cmap='Blues', axis=None, vmin=0, vmax=1)
        .set_caption('Median power relative to 2 tones, mean over the settings of each group'))

The drop has the same shape in both groups, but it is compressed in group B: with 32 tones, the power keeps a larger share of the 2-tone power, as the weakest readings approach the lowest values recorded.

### 4.4 Filtering: how separable are the classes at each setting?
A simple baseline: assign each reading to the class whose median power at the same setting is the closest, and count the readings assigned to their own class. It is not a trained model, but it shows how much the classes overlap at each setting.

In [ ]:
def nearest_median_accuracy(group):
    class_medians = group.groupby('tones')['power_mW'].median()
    predicted = group['power_mW'].map(lambda value: (class_medians - value).abs().idxmin())
    return (predicted == group['tones']).mean()

separability = (power.groupby('setting').apply(nearest_median_accuracy, include_groups=False)
                .rename('Readings assigned to their class (%)').mul(100).to_frame()
                .join(settings.set_index('variable_name')[['dataset_group', 'tx_gain_percent', 'distance_cm']])
                .rename(columns={'dataset_group': 'Group', 'tx_gain_percent': 'Gain (%)', 'distance_cm': 'Distance (cm)'})
                .loc[SETTING_ORDER])
separability.index.name = 'Setting'
display(separability.style.format({'Readings assigned to their class (%)': '{:.0f}'})
        .background_gradient(cmap='Blues', subset=['Readings assigned to their class (%)'], vmin=50, vmax=100)
        .set_caption('Class separability by nearest median'))
print('Mean by group:', separability.groupby('Group')['Readings assigned to their class (%)'].mean().round(0).to_dict())

The classes overlap more as the power decreases: the separability falls from about 90% at full gain and short distance to about 80% at the weakest setting (65% gain). On average it is lower in group B, but the 90% gain settings of group B are as separable as the 75% gain settings of group A.

## 5. Visualization
- **Power by number of tones** at each setting, in mW (log scale) and relative to the 2-tone power: the colour is the transmitter gain, solid lines are group A, dashed lines group B.
- **Distributions by class** at three settings, from the strongest to the weakest: the boxes show how much the classes overlap.

In [ ]:
GAIN_COLORS = {100: '#2a78d6', 75: '#1baf7a', 90: '#eb6834', 65: '#eda100'}
tone_values = list(TONES.values())

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for setting in SETTING_ORDER:
    info = settings.set_index('variable_name').loc[setting]
    style = dict(color=GAIN_COLORS[info['tx_gain_percent']], marker='o', markersize=5, linewidth=1.8,
                 linestyle='-' if info['dataset_group'] == 'A' else '--')
    axes[0].plot(tone_values, medians[setting], **style)
    axes[1].plot(tone_values, relative[setting], **style)
for ax in axes:
    ax.set_xscale('log', base=2)
    ax.set_xticks(tone_values, [str(t) for t in tone_values])
    ax.set_xlabel('Number of tones')
axes[0].set(yscale='log', ylabel='Median received power (mW, log scale)', title='Received power by number of tones')
axes[1].set(ylabel='Power relative to 2 tones', title='Relative to the 2-tone power at the same setting', ylim=(0, 1.05))

handles = [plt.Line2D([], [], color=color, marker='o', label=f'{gain}% gain') for gain, color in GAIN_COLORS.items()]
handles += [plt.Line2D([], [], color='#52514e', linestyle='-', label=GROUPS['A'] + ' (10–20 cm)'),
            plt.Line2D([], [], color='#52514e', linestyle='--', label=GROUPS['B'] + ' (25–35 cm)')]
axes[1].legend(handles=handles, fontsize=9)
plt.tight_layout()
plt.show()

# Distributions by class at the strongest, an intermediate and the weakest setting
examples = ['Gain100_Distance10', 'Gain90_Distance25', 'Gain65_Distance35']
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, setting in zip(axes, examples):
    data = power[power['setting'] == setting]
    group = settings.set_index('variable_name').loc[setting, 'dataset_group']
    sns.boxplot(data=data, x='tones', y='power_mW', color=GAIN_COLORS[int(setting.split('_')[0][4:])],
                fliersize=3, ax=ax)
    ax.set(title=f'{setting} (group {group})', xlabel='Number of tones', ylabel='Received power (mW)')
plt.suptitle('Received power by number of tones, from the strongest to the weakest setting')
plt.tight_layout()
plt.show()

## 6. Conclusion
We loaded the FAIR² multisine SWIPT dataset with the `mlcroissant` library and explored the 750 samples (150 per class) measured at twelve transmitter gain and distance settings:

- **Measurement design**: the settings are split between model development (group A: 100% and 75% gain at 10–20 cm) and external testing (group B: 90% and 65% gain at 25–35 cm); gain and distance are not crossed.
- **Power by number of tones**: at every setting, the received power falls steadily as the number of tones increases, from 2 to 32.
- **Normalized**: relative to the 2-tone power, the drop has a similar shape in both groups, but compressed in the weaker settings of group B.
- **Separability**: a nearest-median baseline assigns most readings to their own class, less often as the power decreases: generalizing from group A to the weaker settings of group B is the harder test this partitioning sets.

Further exploration can train a classifier on group A and evaluate it on group B, as the partitioning intends, or combine the readings of several settings per sample.